In [31]:
import pandas as pd


def ler_planilha(caminho):
    df = pd.read_excel(
        caminho,
        sheet_name="Simples Nacional",  # <-- ADICIONE APENAS ESTA LINHA
        names=["conta contabil", "valor", "classificacao"],
        skiprows=2,
    )

    return df.to_dict(orient="records")


def classificar_conta(contas):

    balanco = {"receita": [], "despesa": [], "outros": []}

    for conta in contas:

        nome = str(conta["classificacao"]).lower()

        if "receita" in nome:
            conta["classificacao"] = "receita"
            balanco["receita"].append(conta)

        elif (
            "despesa" in nome
            or "imposto" in nome
            or "custo" in nome
            or "estorno" in nome
        ):
            conta["classificacao"] = "despesa"
            balanco["despesa"].append(conta)

        else:
            conta["classificacao"] = "outros"
            balanco["outros"].append(conta)

    return balanco


def somar_balanco(balanco):

    total_receita = sum(item["valor"] for item in balanco["receita"])
    total_despesa = sum(item["valor"] for item in balanco["despesa"])
    total_outros = sum(item["valor"] for item in balanco["outros"])

    # --- CALCULAR RECEITA LIQUIDA ---

    prim_receitas = sum(item["valor"] for item in balanco["receita"][:2])
    deducoes_vendas = sum(item["valor"] for item in balanco["despesa"][:2])
    receita_liquida = prim_receitas + deducoes_vendas

    # ---CALCULAR MARGEM DE CONTRIBUIÇÃO ---

    custos_variaveis = sum(item["valor"] for item in balanco["despesa"][2:4])
    margem_contribuicao = receita_liquida + custos_variaveis

    # ---CALCULAR EBTDA ---

    despesas_operacionais = sum(item["valor"] for item in balanco["despesa"][4:15])
    receitas_financeiras = sum(item["valor"] for item in balanco["receita"][2:])
    ebitda = margem_contribuicao + despesas_operacionais + receitas_financeiras + total_outros

    # ---CALCULAR % ---

    margem_ebitda = (ebitda / balanco["receita"][0]["valor"]) * 100

    resultado_operacional = total_receita + total_despesa
    resultado = total_receita + total_despesa + total_outros

    print(f"Receitas : R$ {total_receita:,.2f}")
    print(f"Despesas : R$ {total_despesa:,.2f}")
    print(f"Outros : R$ {total_outros:,.2f}")

    print("\n<==========> NOVOS CÁLCULOS <==========>")
    print(f"Receita Líquida :        R$ {receita_liquida:,.2f}")
    #COLOQUEI BEM ESPAÇADO PRA FICAR CERTINHO
    print(f"Margem de Contribuição : R$ {margem_contribuicao:,.2f}")
    print(f"EBITDA :                 R$ {ebitda:,.2f}")
    print(f"Margem EBITDA :                {margem_ebitda:.1f}%")
    print("<======================================>\n")

    print(f"Resultado operacional : R$ {resultado_operacional:,.2f}")
    print(f"Resultado líquido : R$ {resultado:,.2f}")


def main():

    caminho = "contas.xlsx"

    contas = ler_planilha(caminho)

    balanco = classificar_conta(contas)

    print("Contas :\n")

    for item in contas:
        print(
            f"{item['conta contabil']} | "
            f"R$ {item['valor']} | "
            f"{item['classificacao']}"
        )

    somar_balanco(balanco)


if __name__ == "__main__":
    main()

Contas :

1. Receita de Vendas de Mercadorias | R$ 320000 | receita
2. Receita de Prestação de Serviços | R$ 80000 | receita
3. (-) Devoluções e Abatimentos | R$ -5500 | despesa
4. (-) Simples Nacional sobre Vendas (DAS) | R$ -34000 | despesa
5. (-) Custo das Mercadorias Vendidas (CMV) | R$ -145000 | despesa
6. (-) Custo dos Serviços Prestados (CSP) | R$ -15000 | despesa
7. Pró-labore dos Sócios | R$ -18000 | despesa
8. Salários e Ordenados (Equipe) | R$ -35000 | despesa
9. FGTS (Depósito Mensal) | R$ -2800 | despesa
10. Aluguel e Condomínio | R$ -12000 | despesa
11. Material de Escritório e Limpeza | R$ -1500 | despesa
12. Internet e Telefonia | R$ -850 | despesa
13. Fretes e Entregas (Vendas) | R$ -4200 | despesa
14. Taxas de Cartão e Meios de Pagamento | R$ -6500 | despesa
15. Propaganda e Marketing Digital | R$ -9000 | despesa
16. Honorários Contábeis | R$ -2200 | despesa
17. Manutenção e Reparos | R$ -1800 | despesa
18. Receitas Financeiras (Rendimentos) | R$ 1500 | receita
19. De